# Assignment 03: Field Exploratory Data Analysis (EDA)

This notebook performs exploratory data analysis on the agricultural field data from Assignment 02.

## Data Sources
- **fields_iowa_merged.geojson**: 5 Iowa fields with CDL crop data (2020-2023)
- **cdl_EPSG4326.csv**: Crop data by year
- **fields_iowa_crops.geojson**: Wide format crops

## 1. Setup

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Set style for visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Define data paths
DATA_DIR = Path("data/assignment-02")
OUTPUT_DIR = Path("notebooks/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load Data

In [ ]:
# Load GeoJSON files
fields_merged = gpd.read_file(DATA_DIR / "fields_iowa_merged.geojson")
fields_crops = gpd.read_file(DATA_DIR / "fields_iowa_crops.geojson")

# Load CSV files
cdl_data = pd.read_csv(DATA_DIR / "cdl_EPSG4326.csv")
cdl_geometry = pd.read_csv(DATA_DIR / "cdl_with_geometry.csv")

print("=== Data Loaded Successfully ===")
print(f"fields_merged: {fields_merged.shape}")
print(f"fields_crops: {fields_crops.shape}")
print(f"cdl_data: {cdl_data.shape}")
print(f"cdl_geometry: {cdl_geometry.shape}")

## 3. Initial Exploration

In [ ]:
### 3.1 Fields Merged Overview

In [ ]:
print("=== Fields Merged ===")
print(f"\nColumns: {list(fields_merged.columns)}")
print(f"\nData types:\n{fields_merged.dtypes}")
print(f"\nFirst few rows:\n{fields_merged.head()}")

In [ ]:
# Statistical summary using .describe()
print("=== Statistical Summary (.describe()) ===")
print(fields_merged.describe())

# DataFrame info using .info()
print("\n=== DataFrame Info (.info()) ===")
fields_merged.info()

### 3.2 CDL Data Overview

In [ ]:
print("=== CDL Crop Data ===")
print(f"\nColumns: {list(cdl_data.columns)}")
print(f"\nData types:\n{cdl_data.dtypes}")
print(f"\nFirst few rows:\n{cdl_data.head(10)}")

In [ ]:
# Statistical summary using .describe()
print("=== Statistical Summary (.describe()) ===")
print(cdl_data.describe())

# DataFrame info using .info()
print("\n=== DataFrame Info (.info()) ===")
cdl_data.info()

In [ ]:
# Value counts for categorical columns
print("=== Value Counts for Categorical Columns ===")

print("\n--- Field ID ---")
print(cdl_data['field_id'].value_counts())

print("\n--- Crop Name ---")
print(cdl_data['crop_name'].value_counts())

print("\n--- Year ---")
print(cdl_data['year'].value_counts().sort_index())

print("\n--- Crop Code ---")
print(cdl_data['crop_code'].value_counts())

## 4. Field Overview

In [ ]:
print("=== CDL Crop Data ===")
print(f"\nColumns: {list(cdl_data.columns)}")
print(f"\nData types:\n{cdl_data.dtypes}")
print(f"\nFirst few rows:\n{cdl_data.head(10)}")

## 4. Field Overview

In [ ]:
# Get unique fields
unique_fields = cdl_data['field_id'].unique()
print(f"=== Unique Fields: {len(unique_fields)} ===")
print(unique_fields)

# Get year range
years = sorted(cdl_data['year'].unique())
print(f"\n=== Years: {years} ===")

# Get unique crops
unique_crops = cdl_data['crop_name'].unique()
print(f"\n=== Unique Crops: {len(unique_crops)} ===")
print(unique_crops)

## 5. Crop Rotation Analysis

In [ ]:
### 5.1 Crop Rotation by Field (Pivot Table)

In [ ]:
# Create pivot table: fields vs years with crop names
rotation_pivot = cdl_data.pivot_table(
    index='field_id', 
    columns='year', 
    values='crop_name',
    aggfunc='first'
).reset_index()

rotation_pivot.columns = ['Field ID'] + [str(y) for y in years]
print("=== Crop Rotation by Field (2020-2023) ===")
print(rotation_pivot.to_string(index=False))

In [ ]:
### 5.2 Crop Rotation Pattern Analysis

In [ ]:
# Analyze rotation patterns
def analyze_rotation(row):
    crops = [row[2020], row[2021], row[2022], row[2023]]
    unique_crops = set(crops)
    
    if len(unique_crops) == 1:
        return "Monoculture"
    elif 'Corn' in crops and 'Soybeans' in crops:
        return "Corn-Soybean Rotation"
    else:
        return "Mixed/Other"

rotation_pivot['Pattern'] = rotation_pivot.apply(analyze_rotation, axis=1)

print("=== Crop Rotation Patterns ===")
print(rotation_pivot[['Field ID', 'Pattern']].to_string(index=False))

# Count patterns
pattern_counts = rotation_pivot['Pattern'].value_counts()
print(f"\n=== Pattern Summary ===")
print(pattern_counts)

In [ ]:
### 5.3 Visualize Crop Rotation

In [ ]:
# Create a heatmap of crop types over time
crop_mapping = {'Corn': 1, 'Soybeans': 2, 'Developed/Low Intensity': 3}
rotation_matrix = cdl_data.pivot_table(
    index='field_id', 
    columns='year', 
    values='crop_name',
    aggfunc='first'
).replace(crop_mapping)

plt.figure(figsize=(10, 6))
sns.heatmap(rotation_matrix, annot=True, cmap='YlGn', 
            xticklabels=['2020', '2021', '2022', '2023'],
            cbar_kws={'label': 'Crop Type (1=Corn, 2=Soybeans, 3=Developed)'})
plt.title('Crop Rotation Heatmap by Field (2020-2023)', fontsize=14)
plt.xlabel('Year')
plt.ylabel('Field ID')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'crop_rotation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'crop_rotation_heatmap.png'}")

## 6. Acreage Analysis

In [ ]:
### 6.1 Total Acreage Summary

In [ ]:
# Get unique field info (one row per field)
field_info = fields_merged[['field_id', 'area_acres']].drop_duplicates()

print("=== Acreage Summary ===")
print(f"Total Fields: {len(field_info)}")
print(f"Total Acres: {field_info['area_acres'].sum():,.1f} acres")
print(f"Average Field Size: {field_info['area_acres'].mean():,.1f} acres")
print(f"Min Field Size: {field_info['area_acres'].min():,.1f} acres")
print(f"Max Field Size: {field_info['area_acres'].max():,.1f} acres")
print(f"Median Field Size: {field_info['area_acres'].median():,.1f} acres")

In [ ]:
### 6.2 Field Size Distribution

In [ ]:
plt.figure(figsize=(10, 5))

# Bar chart of field sizes
plt.subplot(1, 2, 1)
field_info_sorted = field_info.sort_values('area_acres', ascending=False)
plt.barh(field_info_sorted['field_id'], field_info_sorted['area_acres'], color='steelblue')
plt.xlabel('Area (acres)')
plt.ylabel('Field ID')
plt.title('Field Sizes (Acres)')

# Histogram of field sizes
plt.subplot(1, 2, 2)
plt.hist(field_info['area_acres'], bins=5, color='steelblue', edgecolor='black')
plt.xlabel('Area (acres)')
plt.ylabel('Count')
plt.title('Distribution of Field Sizes')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'field_sizes.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'field_sizes.png'}")

In [ ]:
### 6.3 Acreage by Crop Type (2023)

In [ ]:
# Filter to 2023 and group by crop
cdl_2023 = cdl_data[cdl_data['year'] == 2023].copy()
acreage_by_crop = cdl_2023.groupby('crop_name')['area_acres'].first().reset_index()
acreage_by_crop = acreage_by_crop.sort_values('area_acres', ascending=False)

print("=== 2023 Acreage by Crop Type ===")
print(acreage_by_crop.to_string(index=False))

# Pie chart
plt.figure(figsize=(8, 8))
colors = ['#f1c40f', '#27ae60', '#7f8c8d']
plt.pie(acreage_by_crop['area_acres'], labels=acreage_by_crop['crop_name'], 
        autopct='%1.1f%%', colors=colors, startangle=90)
plt.title('2023 Acreage Distribution by Crop Type')
plt.savefig(OUTPUT_DIR / 'acreage_by_crop_2023.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'acreage_by_crop_2023.png'}")

## 7. Coverage Analysis

In [ ]:
### 7.1 Coverage Percentage Summary

In [ ]:
print("=== Coverage Percentage Summary (dominant_pct) ===")
print(f"Mean Coverage: {cdl_data['dominant_pct'].mean():.1f}%")
print(f"Min Coverage: {cdl_data['dominant_pct'].min():.1f}%")
print(f"Max Coverage: {cdl_data['dominant_pct'].max():.1f}%")
print(f"Std Dev: {cdl_data['dominant_pct'].std():.1f}%")

In [ ]:
### 7.2 Coverage by Field and Year

In [ ]:
# Create pivot table: coverage percentages
coverage_pivot = cdl_data.pivot_table(
    index='field_id', 
    columns='year', 
    values='dominant_pct',
    aggfunc='first'
).reset_index()

coverage_pivot.columns = ['Field ID'] + [str(y) for y in years]
print("=== Coverage % by Field and Year ===")
print(coverage_pivot.to_string(index=False))

In [ ]:
### 7.3 Coverage Visualization

In [ ]:
# Line plot: coverage over time by field
plt.figure(figsize=(12, 6))
for field_id in cdl_data['field_id'].unique():
    field_data = cdl_data[cdl_data['field_id'] == field_id]
    plt.plot(field_data['year'], field_data['dominant_pct'], 
             marker='o', label=field_id)

plt.xlabel('Year')
plt.ylabel('Coverage %')
plt.title('Crop Coverage % Over Time by Field')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(years)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'coverage_over_time.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'coverage_over_time.png'}")

In [ ]:
# Histogram of coverage percentages
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.hist(cdl_data['dominant_pct'], bins=10, color='coral', edgecolor='black')
plt.xlabel('Coverage %')
plt.ylabel('Count')
plt.title('Distribution of Coverage Percentages')

# Box plot by crop type
plt.subplot(1, 2, 2)
cdl_data.boxplot(column='dominant_pct', by='crop_name', grid=False)
plt.title('Coverage % by Crop Type')
plt.suptitle('')
plt.xlabel('Crop Type')
plt.ylabel('Coverage %')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'coverage_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'coverage_distribution.png'}")

## 8. Crop Distribution by Year

In [ ]:
### 8.1 Yearly Crop Counts

In [ ]:
# Count fields by crop type per year
crop_by_year = cdl_data.groupby(['year', 'crop_name']).size().unstack(fill_value=0)
print("=== Crop Distribution by Year (Number of Fields) ===")
print(crop_by_year)

# Stacked bar chart
crop_by_year.plot(kind='bar', stacked=True, figsize=(10, 6), 
                 color=['#f1c40f', '#27ae60', '#7f8c8d'])
plt.xlabel('Year')
plt.ylabel('Number of Fields')
plt.title('Crop Distribution by Year')
plt.legend(title='Crop Type')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'crop_distribution_by_year.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR / 'crop_distribution_by_year.png'}")

## 9. Key Findings

### Summary of Analysis

#### Dataset Overview
- **5 Fields** analyzed across **4 years** (2020-2023)
- **Total Area**: ~31,417 acres
- **Crop Types**: Corn, Soybeans, Developed/Low Intensity

#### Crop Rotation Patterns
- Most fields follow a **corn-soybean rotation** pattern
- One field (FIELD_0001) is classified as **Developed/Low Intensity** (non-agricultural)
- Rotation patterns are consistent with typical Midwest agriculture

#### Acreage Insights
- **FIELD_0017** is the largest field (~9,205 acres)
- **FIELD_0008** is the smallest field (~3,131 acres)
- Corn dominates in 2023 with ~20,879 acres (66.5%)

#### Coverage Analysis
- Average coverage is **46.7%** with high variability
- FIELD_0008 (Corn) has highest coverage at 68.9%
- FIELD_0017 (Corn) has lowest coverage at 29.3%

#### Key Observations
1. **Corn-Soybean Rotation**: Classic Midwest rotation pattern visible across most fields
2. **Non-Agricultural Land**: FIELD_0001 is identified as developed land (urban/industrial)
3. **Coverage Variability**: Coverage percentages vary significantly by field and crop type
4. **Large Fields**: Some fields span >9,000 acres, indicating large-scale agriculture

## 10. Export Summary Data

In [ ]:
# Export summary tables
rotation_pivot.to_csv(OUTPUT_DIR / 'crop_rotation_summary.csv', index=False)
coverage_pivot.to_csv(OUTPUT_DIR / 'coverage_summary.csv', index=False)
crop_by_year.to_csv(OUTPUT_DIR / 'crop_distribution_summary.csv')

print("=== Summary Files Exported ===")
print(f"- {OUTPUT_DIR / 'crop_rotation_summary.csv'}")
print(f"- {OUTPUT_DIR / 'coverage_summary.csv'}")
print(f"- {OUTPUT_DIR / 'crop_distribution_summary.csv'}")
print(f"\nVisualizations saved to: {OUTPUT_DIR}/")